# 02: Build tables, select columns, join, and filter to Automotive

**Research question:** Can a product's listing details and its early customer reviews predict whether the product will end up poorly rated?

**What this notebook does**
1. Copies both downloaded files into DuckDB tables, `review` and `meta`.
2. Counts rows and previews both tables.
3. Drops the columns the research question doesn't need.
4. Checks that `parent_asin` is safe to join on.
5. Joins every review to its product in a new table, `review_meta`.
6. Keeps only Automotive products.

**Before you run it:** run notebook 01 in the same Colab session, so both files are downloaded and the setup cell (DuckDB, pandas, memory limit) has run.

Every decision in this notebook, with its reason, is listed in [docs/06-decisions.md](../docs/06-decisions.md).

## 1. Copy both files into DuckDB tables

`CREATE TABLE ... AS SELECT` reads a file and stores it as a table inside DuckDB. Unlike the original file, a table can be changed later, e.g. by dropping columns or deleting rows.

- `read_json(...)` reads the zipped JSON Lines file directly.
- `ignore_errors = true` (meta only) tells DuckDB to **skip any line it can't read** instead of stopping. This was needed to load the meta file, but it also means unreadable product lines are skipped silently. Keep it in mind when reporting the product count.

**Where these tables live:** `duckdb.sql(...)` uses DuckDB's default **in-memory** database. The tables sit in RAM and are lost when the session restarts, so this notebook has to be re-run each new session.

Each line takes about a minute or more.

In [ ]:
# Copy the review file into a table called "review" (one row = one review).
duckdb.sql("CREATE TABLE review AS SELECT * FROM read_json('Automotive.jsonl.gz')")

# Copy the meta file into a table called "meta" (one row = one product).
# ignore_errors = true: skip any line that can't be read, instead of stopping.
duckdb.sql("CREATE TABLE meta AS SELECT * FROM read_json('meta_Automotive.jsonl.gz', ignore_errors = true)")

## 2. Preview and count

`LIMIT 5` keeps the preview small. Never call `.df()` on a whole table: it would pull millions of rows into pandas and crash Colab.

In [ ]:
# Show the first 5 reviews as a table, to see the columns.
duckdb.sql("SELECT * FROM review LIMIT 5").df()

In [ ]:
# Show the first 5 products as a table.
duckdb.sql("SELECT * FROM meta LIMIT 5").df()

In [ ]:
# How many reviews?
duckdb.sql("SELECT COUNT(*) AS total_rows FROM review")

In [ ]:
# How many products?
duckdb.sql("SELECT COUNT(*) AS total_rows FROM meta")

**Result:** 19,955,450 reviews and 2,003,129 products.

The review table is about ten times larger because most products have several reviews. `parent_asin` (the product ID) appears in both tables and is chosen as the **join key**.

## 3. Drop the columns the research question doesn't need

`ALTER TABLE ... DROP COLUMN` deletes a column from the table permanently. The original `.gz` file is untouched, so a dropped column can always be re-read from it.

| Table | Column | Reason |
|---|---|---|
| review | `images` | Heavy; photos are outside the research question |
| review | `asin` | Redundant: identifies a variant (colour, size); the research works at product level with `parent_asin` |
| review | `helpful_vote` | **Data leakage**: votes accumulate long after a review is posted |
| meta | `average_rating`, `rating_number` | **Data leakage**: they summarise all reviews, including the later ones the target is built from |
| meta | `images`, `videos` | Heavy; outside the research question |
| meta | `details` | Its useful part (brand) largely duplicates `store`; other keys vary between products |
| meta | `subtitle`, `author` | Undocumented book-page fields, not relevant to the question |

`main_category` is **kept for now**: it is needed to filter to Automotive in section 6.

In [ ]:
# Review table: drop one column per line.
duckdb.sql("ALTER TABLE review DROP COLUMN images")
duckdb.sql("ALTER TABLE review DROP COLUMN asin")
duckdb.sql("ALTER TABLE review DROP COLUMN helpful_vote")

# Meta table: drop one column per line.
duckdb.sql("ALTER TABLE meta DROP COLUMN average_rating")
duckdb.sql("ALTER TABLE meta DROP COLUMN rating_number")
duckdb.sql("ALTER TABLE meta DROP COLUMN images")
duckdb.sql("ALTER TABLE meta DROP COLUMN videos")
duckdb.sql("ALTER TABLE meta DROP COLUMN details")
duckdb.sql("ALTER TABLE meta DROP COLUMN subtitle")
duckdb.sql("ALTER TABLE meta DROP COLUMN author")

## 4. Is `parent_asin` safe to join on?

Three things must hold:
1. No product is missing its `parent_asin` in meta.
2. No `parent_asin` appears twice in meta. Otherwise the join would copy reviews.
3. No review is missing its `parent_asin`.

Repeated `parent_asin` values in the **review** table are expected and fine: one product, many reviews.

In [ ]:
# Check 1: products with no parent_asin in meta. Should be 0.
duckdb.sql("SELECT COUNT(*) AS missing_parent_asin FROM meta WHERE parent_asin IS NULL")

In [ ]:
# Check 2: how many DIFFERENT parent_asin values meta has.
# If this equals the row count (2,003,129), no product appears twice.
duckdb.sql("SELECT COUNT(DISTINCT parent_asin) AS unique_parent_asin FROM meta")

In [ ]:
# Check 3: reviews with no parent_asin. Should be 0.
duckdb.sql("SELECT COUNT(*) AS missing_parent_asin FROM review WHERE parent_asin IS NULL")

In [ ]:
# For context: how many different products were reviewed.
duckdb.sql("SELECT COUNT(DISTINCT parent_asin) AS reviewed_products FROM review")

**Results**

| Check | Result | Meaning |
|---|---|---|
| Missing `parent_asin` in meta | 0 | Every product has an ID |
| Distinct `parent_asin` in meta | 2,003,129 = row count | No product appears twice |
| Missing `parent_asin` in review | 0 | Every review points to a product ID |
| Distinct `parent_asin` in review | 2,003,039 | Many reviews per product, as expected |

The join key is safe on both sides.

## 5. Main categories in meta

The research is about automotive products, but the Automotive file also contains products from other main categories.

In [ ]:
# List every different main_category value in meta.
duckdb.sql("SELECT DISTINCT main_category FROM meta").df()

**Result:** 38 categories plus `None` (no category), 39 values in total. Besides Automotive, these include Industrial & Scientific, AMAZON FASHION, Sports & Outdoors, Books and others.

At product level (counted in an earlier session, see the appendix), Automotive has 1,717,837 of the 2,003,129 products (about 85.8%), and 34,116 products (about 1.7%) have no category.

## 6. Join reviews to products, then filter to Automotive

A **LEFT JOIN** attaches each product's details to every one of its reviews and keeps **every** review, including any whose product isn't in meta (their product columns are left empty, so they can be counted).

- `review AS r`, `meta AS m`: short nicknames; `r.rating` means "rating from the review table".
- `ON r.parent_asin = m.parent_asin`: the matching rule.
- `m.title AS product_title`: both tables have a `title` column, so the product's one is renamed. The review headline keeps the name `title`.

The result is a new table, `review_meta`, with one row per review. Product details repeat for every review of the same product, so this table is large. If DuckDB reports "out of memory", re-run without `m.features`, `m.description` and `m.bought_together`.

In [ ]:
# Build review_meta: every review, with its product's details attached.
join_query = '''
    CREATE TABLE review_meta AS
    SELECT
        r.parent_asin,                   -- product ID (the join key)
        r.rating,                        -- stars the reviewer gave
        r.title,                         -- review headline
        r.text,                          -- review text
        r.user_id,                       -- reviewer ID
        r.timestamp,                     -- when the review was posted (milliseconds since 1970)
        r.verified_purchase,             -- did Amazon confirm the purchase

        m.main_category,                 -- kept for the Automotive filter below
        m.title  AS product_title,       -- product name (renamed to avoid the clash)
        m.price,                         -- product price
        m.store,                         -- store / brand
        m.categories,                    -- category path
        m.features,                      -- bullet-point features
        m.description,                   -- product description
        m.bought_together                -- recommended bundles (to be dropped)
    FROM review AS r
    LEFT JOIN meta AS m
        ON r.parent_asin = m.parent_asin -- match each review to the product with the same ID
'''

# Run the join. This takes several minutes.
duckdb.sql(join_query)

In [ ]:
# The row count must equal the review table's (19,955,450).
duckdb.sql("SELECT COUNT(*) AS joined_rows FROM review_meta")

**Result:** 19,955,450, the same as the review table.

**What this proves:** no review was copied by the join, which confirms there are no duplicate products in meta.
**What it does NOT prove:** that every review found its product. A left join also keeps reviews with no match (with empty product columns), so they are still counted.

In [ ]:
# How many reviews belong to each main category (before filtering).
category_review_query = '''
    SELECT main_category,
           COUNT(*) AS n
    FROM review_meta
    GROUP BY main_category
    ORDER BY n DESC
'''
duckdb.sql(category_review_query).df()

**Result (before filtering):** 16,857,383 reviews belong to Automotive products (about 84.5% of all reviews). The remaining 3,098,067 reviews are 2,912,691 reviews of 38 other categories plus 185,376 reviews with no main category.

In [ ]:
# Delete every review whose product is not Automotive.
# Note: rows with NO main category (NULL) are NOT deleted by this line,
# because SQL treats a comparison with NULL as "unknown", not "true".
duckdb.sql("DELETE FROM review_meta WHERE main_category != 'Automotive'")

**Result after the delete:** `review_meta` holds 16,857,383 Automotive reviews plus 185,376 rows with no main category.

### Open item: rows with no main category (185,376)

These rows mix two cases:
1. **Reviews whose product isn't in meta at all.** No product details exist, so they can't be used.
2. **Reviews of products that exist but have no category label.**

The recommended decision is to remove both, consistent with excluding uncategorised products at product level (about 1.1% of the remaining reviews). The cells below are **not run yet**.

In [ ]:
# NOT RUN YET. Optional: how many of the NULL rows had no product at all (case 1).
# duckdb.sql("SELECT COUNT(*) AS no_product_found FROM review_meta WHERE main_category IS NULL AND product_title IS NULL")

In [ ]:
# NOT RUN YET. Remove the rows with no main category.
# duckdb.sql("DELETE FROM review_meta WHERE main_category IS NULL")

# NOT RUN YET. After that, every row says "Automotive", so the column carries no information.
# duckdb.sql("ALTER TABLE review_meta DROP COLUMN main_category")
# duckdb.sql("ALTER TABLE review_meta DROP COLUMN bought_together")

## Summary

| Step | Result |
|---|---|
| Tables created | `review` (19,955,450 rows), `meta` (2,003,129 rows) |
| Columns dropped | 3 from review, 7 from meta |
| Join key | `parent_asin`: no blanks on either side, no duplicates in meta |
| Joined table | `review_meta`, 19,955,450 rows, no duplicated reviews |
| Automotive filter | 16,857,383 Automotive reviews kept; 2,912,691 removed |
| Open | 185,376 rows with no main category: removal recommended |

**Next:** notebook 03, exploratory data analysis: data quality, distributions, and review timing per product.

---

## Appendix: queries from an earlier session

These were run while deciding what to drop, before the tables above were rebuilt. They are kept for the record.

In [ ]:
# How many DIFFERENT values helpful_vote has (run before the column was dropped).
duckdb.sql("SELECT COUNT(DISTINCT helpful_vote) AS unique_vote FROM review")

**Result:** 741 different values. This measures variety, not how often each value appears. `helpful_vote` was dropped for **data leakage**, not because of this number.

In [ ]:
# Number of PRODUCTS per main category in meta, biggest first.
product_category_query = '''
    SELECT main_category,
           COUNT(*) AS product_count
    FROM meta
    GROUP BY main_category
    ORDER BY product_count DESC
'''
duckdb.sql(product_category_query).df()

**Result (largest groups):**

| main_category | products |
|---|---|
| Automotive | 1,717,837 (about 85.8%) |
| Industrial & Scientific | 58,909 |
| None | 34,116 (about 1.7%) |
| AMAZON FASHION | 26,594 |
| Sports & Outdoors | 19,649 |